# Project 1 — Advanced Solution

Validate a nested dictionary against a template dictionary.

## Rules

- Every template key is required.
- Extra data keys are not permitted.
- Nested dictionaries are validated recursively.
- Leaf values in the template are Python type objects such as `int` or `str`.
- Leaf values in the data must have the **exact** expected type.
- Lists are intentionally outside the scope of this project.
- Validation stops at the first error and reports its dotted path.

The notebook provides both:
1. an exception-based validator (`validate_or_raise`) — the preferred API; and
2. a compatibility wrapper (`validate`) returning `(state, error)`.


## Sample template and data


In [1]:
template = {
    "user_id": int,
    "name": {
        "first": str,
        "last": str,
    },
    "bio": {
        "dob": {
            "year": int,
            "month": int,
            "day": int,
        },
        "birthplace": {
            "country": str,
            "city": str,
        },
    },
}

john = {
    "user_id": 100,
    "name": {
        "first": "John",
        "last": "Cleese",
    },
    "bio": {
        "dob": {
            "year": 1939,
            "month": 11,
            "day": 27,
        },
        "birthplace": {
            "country": "United Kingdom",
            "city": "Weston-super-Mare",
        },
    },
}

eric = {
    "user_id": 101,
    "name": {
        "first": "Eric",
        "last": "Idle",
    },
    "bio": {
        "dob": {
            "year": 1943,
            "month": 3,
            "day": 29,
        },
        "birthplace": {
            "country": "United Kingdom",
        },
    },
}

michael = {
    "user_id": 102,
    "name": {
        "first": "Michael",
        "last": "Palin",
    },
    "bio": {
        "dob": {
            "year": 1943,
            "month": "May",
            "day": 5,
        },
        "birthplace": {
            "country": "United Kingdom",
            "city": "Sheffield",
        },
    },
}


## Exception hierarchy

Custom exceptions make failures machine-readable while still producing the exact human-readable messages requested by the project. Each exception stores the failing `path` separately from its message.


In [2]:
class ValidationError(Exception):
    """Base class for validation failures."""

    label = "validation error"

    def __init__(self, path):
        self.path = path
        super().__init__(self.__str__())

    def __str__(self):
        return "{}: {}".format(self.label, self.path)


class KeyMismatchError(ValidationError):
    """Raised when a required key is absent or an extra key is present."""

    label = "mismatched keys"


class BadTypeError(ValidationError):
    """Raised when a data value has the wrong type."""

    label = "bad type"

    def __init__(self, path, expected=None, actual=None):
        self.expected = expected
        self.actual = actual
        super().__init__(path)


class InvalidTemplateError(ValidationError):
    """Raised when a template leaf is neither a dict nor a type object."""

    label = "invalid template"


## Recursive validator

A few deliberate implementation choices:

- Key equality is checked before descending, so structural errors are detected first.
- Missing keys are reported in template insertion order; extra keys are reported in data insertion order.
- `type(value) is expected_type` is used instead of `isinstance`. This prevents `True` from passing an `int` template because `bool` subclasses `int`.
- The function returns `None` on success and raises on failure, which is idiomatic for an exception-based API.


In [3]:
def _path(parent, key):
    """Join a parent dotted path and one dictionary key."""
    key = str(key)
    return "{}.{}".format(parent, key) if parent else key


def validate_or_raise(data, template, path=""):
    """Validate data against template, raising on the first error."""
    display_path = path or "<root>"

    if not isinstance(template, dict):
        raise InvalidTemplateError(display_path)

    if not isinstance(data, dict):
        raise BadTypeError(
            display_path,
            expected=dict,
            actual=type(data),
        )

    # Check key structure before validating nested values.
    if data.keys() != template.keys():
        # Missing keys: preserve template insertion order.
        for key in template:
            if key not in data:
                raise KeyMismatchError(_path(path, key))

        # Extra keys: preserve data insertion order.
        for key in data:
            if key not in template:
                raise KeyMismatchError(_path(path, key))

    for key, expected in template.items():
        current_path = _path(path, key)
        value = data[key]

        if isinstance(expected, dict):
            validate_or_raise(value, expected, current_path)
            continue

        if not isinstance(expected, type):
            raise InvalidTemplateError(current_path)

        # Exact type comparison is intentional for this project.
        if type(value) is not expected:
            raise BadTypeError(
                current_path,
                expected=expected,
                actual=type(value),
            )


## Compatibility wrapper

This preserves the tuple interface requested in the original prompt while keeping the real validation logic exception-based.


In [4]:
def validate(data, template):
    """Return (True, "") or (False, first_error_message)."""
    try:
        validate_or_raise(data, template)
    except ValidationError as exc:
        return False, str(exc)
    return True, ""


## Required project checks


In [5]:
assert validate(john, template) == (True, "")
assert validate(eric, template) == (
    False,
    "mismatched keys: bio.birthplace.city",
)
assert validate(michael, template) == (
    False,
    "bad type: bio.dob.month",
)

print(validate(john, template))
print(validate(eric, template))
print(validate(michael, template))


(True, '')
(False, 'mismatched keys: bio.birthplace.city')
(False, 'bad type: bio.dob.month')


Expected output:

```text
(True, '')
(False, 'mismatched keys: bio.birthplace.city')
(False, 'bad type: bio.dob.month')
```


## Additional edge-case tests

These cover extra keys, a wrong nested container type, the `bool`/`int` corner case, an invalid template, and an empty dictionary.


In [6]:
# 1) Extra keys are rejected.
john_with_extra_key = {
    **john,
    "active": True,
}
assert validate(john_with_extra_key, template) == (
    False,
    "mismatched keys: active",
)

# 2) A nested object must itself be a dictionary.
wrong_nested_container = {
    **john,
    "name": "John Cleese",
}
assert validate(wrong_nested_container, template) == (
    False,
    "bad type: name",
)

# 3) Exact type semantics: bool must not silently satisfy int.
bool_as_user_id = {
    **john,
    "user_id": True,
}
assert validate(bool_as_user_id, template) == (
    False,
    "bad type: user_id",
)

# 4) Unsupported template leaves are caught explicitly.
bad_template = {
    "user_id": 0,  # should be int, not an example value
}
assert validate({"user_id": 100}, bad_template) == (
    False,
    "invalid template: user_id",
)

# 5) Empty structures are valid when both sides are empty.
assert validate({}, {}) == (True, "")

print("All edge-case tests passed.")


All edge-case tests passed.


## Preferred exception-based usage

In application code, catch specific error classes when different failure types need different handling.


In [7]:
try:
    validate_or_raise(michael, template)
except KeyMismatchError as exc:
    print("Structure error at:", exc.path)
except BadTypeError as exc:
    print(
        "Type error at {}: expected {}, got {}".format(
            exc.path,
            exc.expected.__name__ if exc.expected else "?",
            exc.actual.__name__ if exc.actual else "?",
        )
    )
except InvalidTemplateError as exc:
    print("Template definition error at:", exc.path)


Type error at bio.dob.month: expected int, got str


## Complexity

Let `n` be the total number of keys visited across the nested dictionaries.

- **Time:** `O(n)` in the normal case.
- **Auxiliary space:** `O(d)` for recursion, where `d` is the maximum nesting depth.

For production API validation, a schema library such as `jsonschema`, Pydantic, or Marshmallow is normally preferable because real schemas often need optional fields, lists, unions, ranges, formats, coercion rules, and richer error aggregation.
